# Example: Finite Basis Gaussian

In [ ]:
import numpy as np

# Number of basis functions N.  We use one DC component plus cos/sin pairs,
# so N should be odd: N = 1 + 2*n_freqs, giving frequencies k = 1 … n_freqs.
N = 11
n_freqs = (N - 1) // 2  # highest frequency (cycles per domain)

# Spatial domain: M evenly-spaced sample points on [0, 2π).
M = 256
x = np.linspace(0, 2 * np.pi, M, endpoint=False)

# Build the basis matrix B of shape (M, N).
#   Column 0          : DC component (constant 1)
#   Columns 2k-1, 2k  : cos(k·x) and sin(k·x) for k = 1 … n_freqs
#
# Each column is one basis function evaluated at all sample points.
B = np.zeros((M, N))
B[:, 0] = 1.0  # DC

for k in range(1, n_freqs + 1):
    B[:, 2 * k - 1] = np.cos(k * x)  # cosine at k cycles per domain
    B[:, 2 * k] = np.sin(k * x)  # sine   at k cycles per domain

print(f"Basis matrix shape: {B.shape}  ({M} samples × {N} basis functions)")

In [ ]:
import matplotlib.pyplot as plt

# Plot each basis function as a separate trace, offset vertically so they
# don't overlap.  DC is shown in black; cos/sin pairs share a colour.
fig, ax = plt.subplots(figsize=(10, 7))

offset_step = 2.5  # vertical spacing between traces
cmap = plt.get_cmap("tab10")

for j in range(N):
    offset = (N - 1 - j) * offset_step  # top-to-bottom: DC at top

    if j == 0:
        label = "DC"
        color = "black"
    else:
        k = (j + 1) // 2  # frequency index
        label = f"cos({k}x)" if j % 2 == 1 else f"sin({k}x)"
        color = cmap((k - 1) % 10)

    ax.plot(x, B[:, j] + offset, color=color, lw=1.2, label=label)

ax.set_xlabel("x")
ax.set_xlim(0, 2 * np.pi)
ax.set_xticks([0, np.pi / 2, np.pi, 3 * np.pi / 2, 2 * np.pi])
ax.set_xticklabels(["0", "π/2", "π", "3π/2", "2π"])
ax.set_yticks([])
ax.set_title(f"Sinusoidal basis functions (N={N})")
ax.legend(loc="upper right", fontsize=8, ncol=2)

plt.tight_layout()
plt.show()

In [ ]:
# Prior on basis-function weights.
#
# Each weight w_j ~ N(0, sigma_j^2), where the standard deviation decays
# with the sinusoidal order k of that basis function:
#
#   sigma_j = gamma * exp(-epsilon * k_j)
#
# k_j is 0 for DC, and equals the frequency index k for both cos(kx) and sin(kx).
# With gamma=2 and epsilon=0.5 this gives sigma=2 for DC, ~1.2 at k=1, ~0.74 at k=2, …

gamma = 2.0  # base standard deviation
epsilon = 0.5  # exponential decay rate with sinusoidal order

# Sinusoidal order for each of the N basis functions:
#   j=0 (DC) -> k=0,  j=1,2 (cos/sin at k=1) -> k=1,  j=3,4 -> k=2, …
orders = (np.arange(N) + 1) // 2  # shape (N,)
sigma = gamma * np.exp(-epsilon * orders)  # prior std for each weight

# Build labels for the plot
labels = ["DC"] + [
    f"{'cos' if j % 2 == 1 else 'sin'}({(j + 1) // 2}x)" for j in range(1, N)
]

# Visualise the prior standard deviations
fig, ax = plt.subplots(figsize=(9, 4))
bar_colors = ["black"] + [cmap(((j + 1) // 2 - 1) % 10) for j in range(1, N)]
ax.bar(labels, sigma, color=bar_colors, alpha=0.8, edgecolor="white")
ax.set_ylabel("Prior std  σⱼ")
ax.set_title(f"Prior standard deviations  (γ={gamma}, ε={epsilon})")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

print("orders:", orders)
print("sigma: ", np.round(sigma, 3))

In [ ]:
# Draw sample functions from the prior.
#
# A function is parameterised by a weight vector w of length N.
# The prior is w_j ~ N(0, sigma_j^2), independently for each j.
# The corresponding function is f(x) = B @ w  (shape M,).

rng = np.random.default_rng(seed=42)

n_samples = 5

# Draw weight vectors: each column is one sample, shape (N, n_samples)
W = rng.normal(loc=0.0, scale=sigma[:, np.newaxis], size=(N, n_samples))

# Evaluate the functions at all sample points, shape (M, n_samples)
F = B @ W

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
sample_colors = plt.get_cmap("Set1")(np.linspace(0, 0.8, n_samples))

for s in range(n_samples):
    ax.plot(x, F[:, s], color=sample_colors[s], lw=1.5, label=f"sample {s + 1}")

ax.axhline(0, color="grey", lw=0.8, ls="--")
ax.set_xlabel("x")
ax.set_xlim(0, 2 * np.pi)
ax.set_xticks([0, np.pi / 2, np.pi, 3 * np.pi / 2, 2 * np.pi])
ax.set_xticklabels(["0", "π/2", "π", "3π/2", "2π"])
ax.set_title(f"Prior samples  (N={N}, γ={gamma}, ε={epsilon})")
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ── Parameters ───────────────────────────────────────────────────────────────
noise_std = 0.2  # known measurement noise standard deviation
n_meas = 6  # number of measurements

rng_inf = np.random.default_rng(seed=7)

# ── 1. Draw a true function from the prior ───────────────────────────────────
w_true = rng_inf.normal(0.0, sigma)  # weight vector, shape (N,)
f_true = B @ w_true  # function values at all x, shape (M,)

# ── 2. Simulate noisy measurements ───────────────────────────────────────────
# Pick n_meas random locations on the domain, evaluate the true function there,
# then corrupt each value with independent Gaussian noise.
meas_idx = rng_inf.integers(0, M, size=n_meas)
x_meas = x[meas_idx]  # shape (n_meas,)
y_meas = f_true[meas_idx] + rng_inf.normal(0.0, noise_std, size=n_meas)

# ── 3. Build the design matrix Φ ─────────────────────────────────────────────
# Φ[i, j] = j-th basis function evaluated at measurement location x_meas[i].
# Shape (n_meas, N).  This is the "B matrix" restricted to the measurement points.
Phi = np.zeros((n_meas, N))
Phi[:, 0] = 1.0
for k in range(1, n_freqs + 1):
    Phi[:, 2 * k - 1] = np.cos(k * x_meas)
    Phi[:, 2 * k] = np.sin(k * x_meas)

# ── 4. Gaussian posterior over weights ───────────────────────────────────────
# Prior:      w   ~ N(0, Λ)          where Λ = diag(sigma²)
# Likelihood: y_i = Φ[i,:] @ w + ε,  ε ~ N(0, noise_std²)
#
# Standard Gaussian-Gaussian conjugate update gives:
#   Posterior covariance:  Σ = (Λ⁻¹  +  ΦᵀΦ / noise_std²)⁻¹
#   Posterior mean:        μ = Σ @ (Φᵀ y / noise_std²)

Lambda_inv = np.diag(1.0 / sigma**2)
Sigma_post = np.linalg.inv(Lambda_inv + Phi.T @ Phi / noise_std**2)
mu_post = Sigma_post @ (Phi.T @ y_meas / noise_std**2)

# Evaluate the posterior mean function over the full domain
f_post_mean = B @ mu_post  # shape (M,)

# ── 5. Plot ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(x, f_true, color="steelblue", lw=1.8, label="True function")
ax.plot(x, f_post_mean, color="crimson", lw=1.8, ls="--", label="Posterior mean")
ax.scatter(x_meas, y_meas, color="black", zorder=5, s=60, label="Measurements")

ax.set_xlabel("x")
ax.set_xlim(0, 2 * np.pi)
ax.set_xticks([0, np.pi / 2, np.pi, 3 * np.pi / 2, 2 * np.pi])
ax.set_xticklabels(["0", "π/2", "π", "3π/2", "2π"])
ax.set_title(
    f"Bayesian inference  "
    f"(N={N} basis functions,  {n_meas} measurements,  noise σ={noise_std})"
)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Same true function (w_true / f_true from above), more measurements.
# A separate fixed-seed RNG controls measurement sampling so this cell is
# fully reproducible regardless of what ran before it.

n_meas_20 = 20
rng_20 = np.random.default_rng(seed=99)  # frozen seed → same draws every run

meas_idx_20 = rng_20.integers(0, M, size=n_meas_20)
x_meas_20 = x[meas_idx_20]
y_meas_20 = f_true[meas_idx_20] + rng_20.normal(0.0, noise_std, size=n_meas_20)

# Design matrix and posterior update (same formulas, more rows in Φ)
Phi_20 = np.zeros((n_meas_20, N))
Phi_20[:, 0] = 1.0
for k in range(1, n_freqs + 1):
    Phi_20[:, 2 * k - 1] = np.cos(k * x_meas_20)
    Phi_20[:, 2 * k] = np.sin(k * x_meas_20)

Sigma_post_20 = np.linalg.inv(Lambda_inv + Phi_20.T @ Phi_20 / noise_std**2)
mu_post_20 = Sigma_post_20 @ (Phi_20.T @ y_meas_20 / noise_std**2)
f_post_mean_20 = B @ mu_post_20

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

cases = [
    (n_meas, x_meas, y_meas, f_post_mean),
    (n_meas_20, x_meas_20, y_meas_20, f_post_mean_20),
]
for ax, (nm, xm, ym, fpm) in zip(axes, cases, strict=True):
    ax.plot(x, f_true, color="steelblue", lw=1.8, label="True function")
    ax.plot(x, fpm, color="crimson", lw=1.8, ls="--", label="Posterior mean")
    ax.scatter(xm, ym, color="black", zorder=5, s=50, label=f"{nm} measurements")
    ax.set_xlim(0, 2 * np.pi)
    ax.set_xticks([0, np.pi / 2, np.pi, 3 * np.pi / 2, 2 * np.pi])
    ax.set_xticklabels(["0", "π/2", "π", "3π/2", "2π"])
    ax.set_title(f"{nm} measurements  (noise σ={noise_std})")
    ax.legend(fontsize=8)

fig.suptitle(f"Homing in — N={N} basis functions", fontsize=13)
plt.tight_layout()
plt.show()